# Notebook 3: Generate Reports

**VLM-ARB Cloud Framework**

This notebook fetches results from Firestore and generates comprehensive reports with visualizations.

## Workflow
1. Setup: Authenticate with Firebase
2. Fetch evaluation results from Firestore
3. Generate visualizations (bar charts, heatmaps)
4. Create PDF reports
5. Upload reports to Cloud Storage
6. Generate shareable links for team

**Key Feature**: Shareable public links - all team members can view reports without additional auth.

---

## Cell 1: Install Dependencies & Setup

In [ ]:
# Install dependencies
import subprocess
import sys

packages = [
    'firebase-admin',
    'matplotlib',
    'seaborn',
    'numpy',
    'pandas',
    'reportlab',
    'pillow',
]

for pkg in packages:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])

print("✅ All dependencies installed")

## Cell 2: Setup Authentication & Fetch Results

In [ ]:
## Cell 2: Setup & Fetch Results

import sys
sys.path.insert(0, '/root')

from pathlib import Path
from utilities.auth_helpers import quick_colab_setup
from utilities.cloud_sync import FirebaseSync, validate_firebase_credentials
from datetime import datetime
import logging

# Quick setup
team_folder, context, gpu_info = quick_colab_setup()

# Initialize Firebase
creds_path = context['creds_path']
try:
    fs = FirebaseSync(creds_path)
    print("✅ Firebase initialized")
except:
    fs = None
    print("⚠️  Firebase unavailable - will use Firestore fallback only")

# Fetch all completed runs from Firestore
print("\n📊 Fetching evaluation results...")

runs = []
if fs and not fs.offline_mode:
    try:
        runs = fs.list_runs(limit=50)
        print(f"✅ Found {len(runs)} completed evaluations")
    except:
        print("⚠️  Could not fetch from Firestore - showing sample data")
else:
    print("⚠️  Using sample data (Firestore unavailable)")

# If no runs found, create sample data for demo
if not runs:
    from datetime import datetime, timedelta
    sample_run = {
        'run_id': f'eval_sample_{datetime.now().strftime("%Y%m%d_%H%M%S")}',
        'timestamp': datetime.now().isoformat(),
        'metrics': {
            'clip': {'asr': 0.35, 'ods': 0.28, 'sbr': 0.0},
            'mobilevit': {'asr': 0.45, 'ods': 0.38, 'sbr': 0.0},
            'blip2': {'asr': 0.68, 'ods': 0.58, 'sbr': 0.05},
        }
    }
    runs = [sample_run]
    print("\n⚠️  Using sample data for demo report")

print(f"\n📋 Available Runs: {len(runs)}")

## Cell 3: Select Run(s) & Display Results

In [ ]:
import pandas as pd

# Auto-select latest run (or you can manually select)
if runs:
    selected_run = runs[0]  # Latest run
    selected_run_id = selected_run.get('run_id', 'unknown')
    
    print(f"\n📊 Selected Run: {selected_run_id}")
    
    # Extract metrics
    metrics = selected_run.get('metrics', {})
    print(f"\n📈 Results Summary:")
    
    # Create DataFrame for display
    results_data = []
    for model_name, model_metrics in metrics.items():
        if isinstance(model_metrics, dict) and 'asr' in model_metrics:
            results_data.append({
                'Model': model_name,
                'ASR': f"{model_metrics.get('asr', 0):.3f}",
                'ODS': f"{model_metrics.get('ods', 0):.3f}",
                'SBR': f"{model_metrics.get('sbr', 0):.3f}",
                'CMCS': f"{model_metrics.get('cmcs', 'N/A')}"
            })
    
    if results_data:
        df = pd.DataFrame(results_data)
        print(df.to_string(index=False))
    else:
        print("No model results in selected run")
else:
    print("No runs available")
    selected_run = None

## Cell 4: Generate Visualizations (Bar Charts)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from pathlib import Path

if selected_run:
    # Setup output directory
    figures_dir = Path("/tmp/vlm_arb_figures")
    figures_dir.mkdir(exist_ok=True)
    
    # Extract model names and metrics
    metrics = selected_run.get('metrics', {})
    models = []
    asr_values = []
    ods_values = []
    sbr_values = []
    
    for model_name, model_metrics in metrics.items():
        if isinstance(model_metrics, dict) and 'asr' in model_metrics:
            models.append(model_name)
            asr_values.append(model_metrics.get('asr', 0))
            ods_values.append(model_metrics.get('ods', 0))
            sbr_values.append(model_metrics.get('sbr', 0))
    
    if models:
        # Create 3-panel comparison chart
        fig, axes = plt.subplots(1, 3, figsize=(15, 4))
        fig.suptitle(f'VLM-ARB Model Robustness: {selected_run_id}', fontsize=14, fontweight='bold')
        
        # ASR (Attack Success Rate)
        axes[0].bar(models, asr_values, color='#FF6B6B')
        axes[0].set_title('Attack Success Rate (ASR)')
        axes[0].set_ylabel('ASR Score')
        axes[0].set_ylim([0, 1.0])
        axes[0].tick_params(axis='x', rotation=45)
        
        # ODS (Output Deviation Score)
        axes[1].bar(models, ods_values, color='#4ECDC4')
        axes[1].set_title('Output Deviation Score (ODS)')
        axes[1].set_ylabel('ODS Score')
        axes[1].set_ylim([0, 1.0])
        axes[1].tick_params(axis='x', rotation=45)
        
        # SBR (Safety Bypass Rate)
        axes[2].bar(models, sbr_values, color='#95E1D3')
        axes[2].set_title('Safety Bypass Rate (SBR)')
        axes[2].set_ylabel('SBR Score')
        axes[2].set_ylim([0, 1.0])
        axes[2].tick_params(axis='x', rotation=45)
        
        plt.tight_layout()
        
        # Save figure
        chart_path = figures_dir / "model_comparison.png"
        plt.savefig(chart_path, dpi=150, bbox_inches='tight')
        print(f"✅ Chart saved: {chart_path}")
        plt.close()
    else:
        print("⚠️  No model metrics found in results")
else:
    print("⏭️  Skipping visualization (no selected run)")

## Cell 5: Create PDF Report

In [ ]:
from pathlib import Path
import json
import shutil

# Create reports directory on Google Drive
drive_reports_dir = Path("/content/drive/MyDrive/VLM-ARB-Team/reports")
drive_reports_dir.mkdir(parents=True, exist_ok=True)

if pdf_path and pdf_path.exists():
    print("💾 Saving report to Google Drive...")
    
    # Copy PDF to Drive
    drive_pdf_path = drive_reports_dir / f"{selected_run_id}_report.pdf"
    shutil.copy(str(pdf_path), str(drive_pdf_path))
    print(f"✅ PDF saved: {drive_pdf_path}")
    
    # Copy chart image to Drive
    chart_path = figures_dir / "model_comparison.png"
    if chart_path.exists():
        drive_chart_path = drive_reports_dir / f"{selected_run_id}_chart.png"
        shutil.copy(str(chart_path), str(drive_chart_path))
        print(f"✅ Chart saved: {drive_chart_path}")
    
    # Copy JSON results to Drive
    json_path = Path("/tmp/eval_results.json")
    if not json_path.exists():
        with open(json_path, 'w') as f:
            json.dump(metrics, f, indent=2, default=str)
    
    drive_json_path = drive_reports_dir / f"{selected_run_id}_results.json"
    shutil.copy(str(json_path), str(drive_json_path))
    print(f"✅ JSON saved: {drive_json_path}")
    
    print(f"\n📂 All reports saved to Google Drive:")
    print(f"   {drive_reports_dir}")
    
else:
    print("⏭️  No PDF to save")

## Cell 6: Upload Reports to Cloud Storage

In [ ]:
print("\n" + "="*60)
print("✅ REPORT GENERATION COMPLETE")
print("="*60)

drive_reports_dir = Path("/content/drive/MyDrive/VLM-ARB-Team/reports")

if drive_reports_dir.exists():
    print(f"\n📂 Reports saved to Google Drive:")
    print(f"   📁 {drive_reports_dir}")
    
    print(f"\n📋 Report Files:")
    for report_file in sorted(drive_reports_dir.glob(f"{selected_run_id}*")):
        print(f"   📄 {report_file.name}")
    
    print(f"\n🔗 To share reports with team:")
    print(f"   1. Open Google Drive")
    print(f"   2. Navigate to VLM-ARB-Team/reports")
    print(f"   3. Right-click on PDF file → Share")
    print(f"   4. Set access to 'Anyone with link' or 'Your team'")
    print(f"   5. Copy link and share via Slack/email")
    
    print(f"\n💡 Tip: Share the entire 'reports' folder with your team")
    print(f"   for easy access to all historical reports")
else:
    print("⚠️  Reports directory not found")

print("="*60)

## Cell 7: Generate Shareable Links & Display Results

In [ ]:
print("\n" + "="*60)
print("✅ REPORT GENERATION COMPLETE")
print("="*60)

if selected_run:
    print(f"\n📊 Report Details:")
    print(f"   Run ID: {selected_run_id}")
    print(f"   Timestamp: {datetime.now().isoformat()}")
    print(f"   Models: {len(models)}")
    
    if pdf_url:
        print(f"\n🔗 Shareable Links (paste in team chat/email):")
        print(f"   PDF Report: {pdf_url}")
        if chart_url:
            print(f"   Chart: {chart_url}")
        if json_url:
            print(f"   Results (JSON): {json_url}")
    else:
        print(f"\n📂 Local Files:")
        print(f"   PDF: {pdf_path}")
        print(f"   Chart: {figures_dir}/model_comparison.png")
    
    print(f"\n📊 Metrics Summary:")
    for model_name in models:
        model_metrics = metrics.get(model_name, {})
        print(f"   {model_name}: ASR={model_metrics.get('asr', 0):.3f}, ODS={model_metrics.get('ods', 0):.3f}, SBR={model_metrics.get('sbr', 0):.3f}")
else:
    print("No results to display")

print("="*60)

## Cell 8: Optional - Browse All Historical Reports

In [ ]:
print("\n📚 All Generated Reports:")
print("\nTo view all reports, fetch from Cloud Storage:")

if fs and not fs.offline_mode:
    all_reports = fs.list_files("reports/")
    print(f"\nFound {len(all_reports)} files in Cloud Storage:")
    for report_file in all_reports[:10]:  # Show first 10
        file_name = Path(report_file).name
        url = fs.get_public_url(report_file)
        print(f"   • {file_name}")
else:
    print("Not connected to Cloud Storage")